In [ ]:
!pip install datasets transformers peft

In [ ]:
import torch
import re
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# =====================================================
# CONFIG
# =====================================================

MODEL = "microsoft/phi-3-mini-4k-instruct"

LORA_PATH = ("/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi_grpo_2/phi3_grpo_best")

NUM_SAMPLES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# LOAD MODEL
# =====================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16
).to(DEVICE)

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

model.eval()

# =====================================================
# PROMPTS
# =====================================================

def build_reasoning_prompt(q):
    return f"""You are a reasoning model.

STRICT FORMAT:
<think>...</think>
<answer>...</answer>

Question: {q}

<think>
"""

def build_mmlu_prompt(question, choices):
    return f"""Answer ONLY with a single letter: A, B, C, or D.

Question:
{question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Answer:
"""

# =====================================================
# EXTRACTION
# =====================================================

def extract_number(text):

    answer_match = re.search(
        r"<answer>\s*(-?[\d,]+)",
        text,
        re.IGNORECASE
    )

    if answer_match:
        return answer_match.group(1).replace(",", "")

    hash_match = re.findall(
        r"####\s*(-?[\d,]+)",
        text
    )

    if hash_match:
        return hash_match[-1].replace(",", "")

    nums = re.findall(
        r"-?[\d,]+",
        text
    )

    if nums:
        return nums[-1].replace(",", "")

    return None


def extract_yes_no(text):

    t = text.lower()

    if "yes" in t:
        return "yes"

    if "no" in t:
        return "no"

    return None


def extract_mcq(text):

    text = text.upper()

    patterns = [
        r"ANSWER\s*[:\-]?\s*([ABCD])",
        r"<ANSWER>\s*([ABCD])",
        r"\b([ABCD])\b\s*$"
    ]

    for p in patterns:
        matches = re.findall(p, text)

        if matches:
            return matches[-1]

    letters = re.findall(r"\b([ABCD])\b", text)

    if letters:
        return letters[-1]

    return None

# =====================================================
# GENERATION
# =====================================================

def generate_reasoning(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    generated_response = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    )

    return full_output, generated_response


def generate_mcq(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    generated_response = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    )

    return full_output, generated_response

# =====================================================
# GSM8K
# =====================================================

gsm_rows = []

gsm8k = load_dataset(
    "gsm8k",
    "main",
    split="test"
).select(range(NUM_SAMPLES))

gsm_correct = 0

for sample in gsm8k:

    q = sample["question"]

    gt = (
        sample["answer"]
        .split("####")[-1]
        .strip()
    )

    prompt = build_reasoning_prompt(q)

    full_output, generated_response = generate_reasoning(
        prompt
    )

    pred = extract_number(generated_response)

    is_correct = pred == gt

    if is_correct:
        gsm_correct += 1

    gsm_rows.append({
        "question": q,
        "ground_truth": gt,
        "prediction": pred,
        "prompt": prompt,
        "generated_response": generated_response,
        "full_output": full_output,
        "correct": is_correct
    })
    print(
        f"GSM8K Progress: {len(gsm_rows)}/{NUM_SAMPLES}",
        end="\r"
    )

gsm_acc = gsm_correct / NUM_SAMPLES

pd.DataFrame(gsm_rows).to_csv(
    "/kaggle/working/gsm8k_predictions.csv",
    index=False
)

# =====================================================
# StrategyQA
# =====================================================

qa_rows = []

strategyqa = load_dataset(
    "ChilleD/StrategyQA",
    split="test"
).select(range(NUM_SAMPLES))

qa_correct = 0

for sample in strategyqa:

    q = sample["question"]

    gt = (
        "yes"
        if sample["answer"]
        else "no"
    )

    prompt = build_reasoning_prompt(q)

    full_output, generated_response = generate_reasoning(
        prompt
    )

    pred = extract_yes_no(generated_response)

    is_correct = pred == gt

    if is_correct:
        qa_correct += 1

    qa_rows.append({
        "question": q,
        "ground_truth": gt,
        "prediction": pred,
        "prompt": prompt,
        "generated_response": generated_response,
        "full_output": full_output,
        "correct": is_correct
    })

qa_acc = qa_correct / NUM_SAMPLES

pd.DataFrame(qa_rows).to_csv(
    "/kaggle/working/strategyqa_predictions.csv",
    index=False
)

# =====================================================
# MMLU
# =====================================================

subjects = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies"
]

mmlu_rows = []

answer_dist = {
    "A": 0,
    "B": 0,
    "C": 0,
    "D": 0,
    "None": 0
}

mmlu_correct = 0
total = 0

for sub in subjects:

    ds = load_dataset(
        "cais/mmlu",
        sub,
        split="test"
    )

    ds = ds.select(
        range(
            min(NUM_SAMPLES, len(ds))
        )
    )

    for sample in ds:

        q = sample["question"]
        choices = sample["choices"]

        gt = chr(
            65 + sample["answer"]
        )

        prompt = build_mmlu_prompt(
            q,
            choices
        )

        full_output, generated_response = generate_mcq(
            prompt
        )

        pred = extract_mcq(
            generated_response
        )

        if pred is None:
            answer_dist["None"] += 1
        else:
            answer_dist[pred] += 1

        is_correct = pred == gt

        if is_correct:
            mmlu_correct += 1

        total += 1

        mmlu_rows.append({
            "subject": sub,
            "question": q,
            "choices": str(choices),
            "ground_truth": gt,
            "prediction": pred,
            "prompt": prompt,
            "generated_response": generated_response,
            "full_output": full_output,
            "correct": is_correct
        })

mmlu_acc = mmlu_correct / total

pd.DataFrame(mmlu_rows).to_csv(
    "/kaggle/working/mmlu_predictions.csv",
    index=False
)

# =====================================================
# RESULTS
# =====================================================

print("\n===== FINAL RESULTS =====")
print(f"GSM8K: {gsm_acc:.3f}")
print(f"StrategyQA: {qa_acc:.3f}")
print(f"MMLU: {mmlu_acc:.3f}")

print("\n===== MMLU ANSWER DISTRIBUTION =====")

for k, v in answer_dist.items():
    print(f"{k}: {v}")

print("\nCSV FILES SAVED:")
print("/kaggle/working/gsm8k_predictions.csv")
print("/kaggle/working/strategyqa_predictions.csv")
print("/kaggle/working/mmlu_predictions.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/mmlu_predictions.csv")

for i in range(10):
    print("="*80)
    print(df.iloc[i]["raw_model_output"])